In [1]:
"""
Regression tests for Type-II NUFFT low-rank implementation, with improved reporting.

Features
--------
- Data-driven test definitions (easy to add many tests).
- Aggregated results table (PASS/FAIL, rel_l2, max_abs, timing).
- Optional plots (error bars + scatter).
- Optional JSON report (CI-friendly).
- Optional "run all then fail" behavior (default).
- Jupyter-safe argument parsing (ignores IPython-injected args like "-f kernel.json").

Usage
-----
python test_nufft2_lowrank_vs_dense_pretty.py
python test_nufft2_lowrank_vs_dense_pretty.py --plot
python test_nufft2_lowrank_vs_dense_pretty.py --json-out results.json
python test_nufft2_lowrank_vs_dense_pretty.py --repeat 3
python test_nufft2_lowrank_vs_dense_pretty.py --fail-fast
"""

import _init_path

from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Tuple, Any
import argparse
import json
import time

import numpy as np

from classical.nudft_II import nudft_type2_dense
from classical.nufft_II_lowrank import nufft2

from datasets.data import (
    nodes_uniform,
    nodes_random,
    nodes_perturbed_uniform,
    nodes_near_colliding,
    nodes_clustered,
)
from datasets.signals import signal_random_complex
from utils import rel_l2, max_abs


# ---------------------------------------------------------------------
# Global configuration defaults
# ---------------------------------------------------------------------
EPS_DEFAULT: float = 1e-15
REL_TOL_DEFAULT: float = 5e-12
ABS_TOL_DEFAULT: float = 5e-12
N_DEFAULT: int = 2**8
SEED_DEFAULT: int = 0


# ---------------------------------------------------------------------
# Result model
# ---------------------------------------------------------------------
@dataclass
class CheckResult:
    name: str
    rel_l2: float
    max_abs: float
    rel_tol: float
    abs_tol: float
    ok: bool
    t_test_s: float
    t_ref_s: float
    notes: str = ""


@dataclass
class SuiteResult:
    results: List[CheckResult]

    @property
    def n_total(self) -> int:
        return len(self.results)

    @property
    def n_failed(self) -> int:
        return sum(1 for r in self.results if not r.ok)

    @property
    def n_passed(self) -> int:
        return self.n_total - self.n_failed


# ---------------------------------------------------------------------
# Utilities: printing / formatting
# ---------------------------------------------------------------------
def _fmt_float(x: float) -> str:
    if np.isnan(x):
        return "nan"
    if x == 0.0:
        return "0"
    return f"{x:.3e}"


def print_table(results: List[CheckResult]) -> None:
    headers = ["#", "Test", "PASS", "rel_l2", "max_abs", "rel_tol", "abs_tol", "t_test(s)", "t_ref(s)", "Notes"]
    rows: List[List[str]] = []
    for i, r in enumerate(results, 1):
        rows.append([
            str(i),
            r.name,
            "YES" if r.ok else "NO",
            _fmt_float(r.rel_l2),
            _fmt_float(r.max_abs),
            _fmt_float(r.rel_tol),
            _fmt_float(r.abs_tol),
            f"{r.t_test_s:.4f}",
            f"{r.t_ref_s:.4f}",
            r.notes,
        ])

    widths = [len(h) for h in headers]
    for row in rows:
        for j, cell in enumerate(row):
            widths[j] = max(widths[j], len(cell))

    def line() -> str:
        return "-+-".join("-" * (w + 2) for w in widths)

    def format_row(row: List[str]) -> str:
        return " | ".join(f" {cell:<{widths[j]}} " for j, cell in enumerate(row))

    print(format_row(headers))
    print(line())
    for row in rows:
        print(format_row(row))


def to_jsonable(results: List[CheckResult]) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for r in results:
        out.append({
            "name": r.name,
            "rel_l2": float(r.rel_l2),
            "max_abs": float(r.max_abs),
            "rel_tol": float(r.rel_tol),
            "abs_tol": float(r.abs_tol),
            "ok": bool(r.ok),
            "t_test_s": float(r.t_test_s),
            "t_ref_s": float(r.t_ref_s),
            "notes": r.notes,
        })
    return out


# ---------------------------------------------------------------------
# Core check: evaluate test + reference, compute errors and timing
# ---------------------------------------------------------------------
def run_check(
    name: str,
    make_data: Callable[[], Tuple[np.ndarray, np.ndarray]],
    test_fn: Callable[[np.ndarray, np.ndarray], np.ndarray],
    ref_fn: Callable[[np.ndarray, np.ndarray], np.ndarray],
    rel_tol: float,
    abs_tol: float,
    repeat: int = 1,
    notes: str = "",
) -> CheckResult:
    # Data is generated once per check to keep comparability between methods.
    x, c = make_data()

    # Timings: run multiple times and take minimum to reduce noise.
    f_test: Optional[np.ndarray] = None
    t_test_best = float("inf")
    for _ in range(max(1, repeat)):
        t0 = time.perf_counter()
        f_test = test_fn(x, c)
        t1 = time.perf_counter()
        t_test_best = min(t_test_best, t1 - t0)

    f_ref: Optional[np.ndarray] = None
    t_ref_best = float("inf")
    for _ in range(max(1, repeat)):
        t0 = time.perf_counter()
        f_ref = ref_fn(x, c)
        t1 = time.perf_counter()
        t_ref_best = min(t_ref_best, t1 - t0)

    assert f_test is not None and f_ref is not None

    r = rel_l2(f_test, f_ref)
    m = max_abs(f_test, f_ref)
    ok = (r <= rel_tol) and (m <= abs_tol)

    return CheckResult(
        name=name,
        rel_l2=float(r),
        max_abs=float(m),
        rel_tol=float(rel_tol),
        abs_tol=float(abs_tol),
        ok=bool(ok),
        t_test_s=float(t_test_best),
        t_ref_s=float(t_ref_best),
        notes=notes,
    )


# ---------------------------------------------------------------------
# Plotting (optional)
# ---------------------------------------------------------------------
def plot_results(results: List[CheckResult], out_prefix: Optional[str] = None) -> None:
    import matplotlib.pyplot as plt

    rels = np.array([r.rel_l2 for r in results], dtype=float)
    maxs = np.array([r.max_abs for r in results], dtype=float)
    oks = np.array([r.ok for r in results], dtype=bool)
    x = np.arange(len(results))

    # Bar chart: rel_l2
    plt.figure()
    plt.bar(x, rels)
    plt.yscale("log")
    plt.xticks(x, [str(i + 1) for i in range(len(results))], rotation=0)
    plt.xlabel("Test #")
    plt.ylabel("rel_l2 (log scale)")
    plt.title("Relative L2 error per test")
    for i, ok in enumerate(oks):
        if not ok:
            plt.text(i, rels[i], " FAIL", rotation=90, va="bottom")
    plt.tight_layout()
    if out_prefix:
        plt.savefig(f"{out_prefix}_rel_l2.png", dpi=200)

    # Bar chart: max_abs
    plt.figure()
    plt.bar(x, maxs)
    plt.yscale("log")
    plt.xticks(x, [str(i + 1) for i in range(len(results))], rotation=0)
    plt.xlabel("Test #")
    plt.ylabel("max_abs (log scale)")
    plt.title("Max absolute error per test")
    for i, ok in enumerate(oks):
        if not ok:
            plt.text(i, maxs[i], " FAIL", rotation=90, va="bottom")
    plt.tight_layout()
    if out_prefix:
        plt.savefig(f"{out_prefix}_max_abs.png", dpi=200)

    # Scatter: rel_l2 vs max_abs
    plt.figure()
    plt.scatter(rels, maxs)
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("rel_l2 (log)")
    plt.ylabel("max_abs (log)")
    plt.title("Error scatter: rel_l2 vs max_abs")
    for i, ok in enumerate(oks):
        if not ok:
            plt.annotate(str(i + 1), (rels[i], maxs[i]))
    plt.tight_layout()
    if out_prefix:
        plt.savefig(f"{out_prefix}_scatter.png", dpi=200)

    if not out_prefix:
        plt.show()


# ---------------------------------------------------------------------
# Test suite definition
# ---------------------------------------------------------------------
def build_suite(
    N: int,
    eps: float,
    rng: np.random.Generator,
    rel_tol: float,
    abs_tol: float,
) -> List[Dict[str, Any]]:
    """
    Return a list of check specifications. Each check spec is a dict with:
      - name
      - make_data
      - test_fn
      - ref_fn
      - rel_tol, abs_tol
      - notes (optional)
    """

    def lr(x: np.ndarray, c: np.ndarray) -> np.ndarray:
        return nufft2(c, x, eps)

    def dense(x: np.ndarray, c: np.ndarray) -> np.ndarray:
        return nudft_type2_dense(x, c)

    suite: List[Dict[str, Any]] = []

    # A) Uniform grid sanity: equals FFT
    suite.append(dict(
        name="uniform: lowrank vs dense",
        make_data=lambda: (nodes_uniform(N), signal_random_complex(N, rng=rng)),
        test_fn=lr,
        ref_fn=dense,
        rel_tol=rel_tol,
        abs_tol=abs_tol,
        notes="sanity",
    ))

    suite.append(dict(
        name="uniform: dense vs fft",
        make_data=lambda: (nodes_uniform(N), signal_random_complex(N, rng=rng)),
        test_fn=lambda x, c: dense(x, c),
        ref_fn=lambda x, c: np.fft.fft(c),
        rel_tol=1e-11,
        abs_tol=1e-11,
        notes="FFT equivalence",
    ))

    suite.append(dict(
        name="uniform: lowrank vs fft",
        make_data=lambda: (nodes_uniform(N), signal_random_complex(N, rng=rng)),
        test_fn=lambda x, c: lr(x, c),
        ref_fn=lambda x, c: np.fft.fft(c),
        rel_tol=rel_tol,
        abs_tol=abs_tol,
        notes="FFT equivalence",
    ))

    # B) Random nonuniform nodes: lowrank vs dense
    suite.append(dict(
        name="random nodes: lowrank vs dense",
        make_data=lambda: (nodes_random(N, rng=rng), signal_random_complex(N, rng=rng)),
        test_fn=lr,
        ref_fn=dense,
        rel_tol=rel_tol,
        abs_tol=abs_tol,
    ))

    # C) Perturbed uniform nodes (small jitter): lowrank vs dense
    suite.append(dict(
        name="perturbed uniform: lowrank vs dense",
        make_data=lambda: (nodes_perturbed_uniform(N, jitter=0.2 / N, rng=rng),
                           signal_random_complex(N, rng=rng)),
        test_fn=lr,
        ref_fn=dense,
        rel_tol=rel_tol,
        abs_tol=abs_tol,
    ))

    # D) Known-answer: impulse response
    def make_impulse_data() -> Tuple[np.ndarray, np.ndarray]:
        x = nodes_random(N, rng=rng)
        c = np.zeros(N, dtype=complex)
        c[7] = 1.0 + 0.0j
        return x, c

    def expected_impulse(x: np.ndarray, c: np.ndarray) -> np.ndarray:
        j0 = int(np.argmax(np.abs(c)))
        k = np.arange(N, dtype=float)
        return np.exp(-2j * np.pi * x[j0] * k)

    suite.append(dict(
        name="impulse: dense vs expected",
        make_data=make_impulse_data,
        test_fn=lambda x, c: dense(x, c),
        ref_fn=lambda x, c: expected_impulse(x, c),
        rel_tol=1e-13,
        abs_tol=1e-12,
        notes="known answer",
    ))

    suite.append(dict(
        name="impulse: lowrank vs expected",
        make_data=make_impulse_data,
        test_fn=lambda x, c: lr(x, c),
        ref_fn=lambda x, c: expected_impulse(x, c),
        rel_tol=rel_tol,
        abs_tol=abs_tol,
        notes="known answer",
    ))

    # E) “Harder” geometries: near-colliding / clustered
    suite.append(dict(
        name="near-colliding: lowrank vs dense",
        make_data=lambda: (nodes_near_colliding(N, min_sep=1e-4, rng=rng),
                           signal_random_complex(N, rng=rng)),
        test_fn=lr,
        ref_fn=dense,
        rel_tol=rel_tol,
        abs_tol=abs_tol,
        notes="hard geometry",
    ))

    suite.append(dict(
        name="clustered: lowrank vs dense",
        make_data=lambda: (nodes_clustered(N, n_clusters=3, cluster_std=0.01, rng=rng),
                           signal_random_complex(N, rng=rng)),
        test_fn=lr,
        ref_fn=dense,
        rel_tol=rel_tol,
        abs_tol=abs_tol,
        notes="hard geometry",
    ))

    return suite


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------
def main() -> None:
    parser = argparse.ArgumentParser(add_help=True)
    parser.add_argument("--N", type=int, default=N_DEFAULT)
    parser.add_argument("--eps", type=float, default=EPS_DEFAULT)
    parser.add_argument("--rel-tol", type=float, default=REL_TOL_DEFAULT)
    parser.add_argument("--abs-tol", type=float, default=ABS_TOL_DEFAULT)
    parser.add_argument("--seed", type=int, default=SEED_DEFAULT)
    parser.add_argument("--repeat", type=int, default=1, help="Repeat each method run and take min time.")
    parser.add_argument("--fail-fast", action="store_true", help="Stop at first failure.")
    parser.add_argument("--plot", action="store_true", help="Show/save plots.")
    parser.add_argument("--plot-prefix", type=str, default=None, help="If set, save plots to files with this prefix.")
    parser.add_argument("--json-out", type=str, default=None, help="Write machine-readable results JSON.")

    # Jupyter/IPython injects extra CLI args (e.g. "-f kernel.json").
    # parse_known_args() ignores unknown flags so the script runs both in CLI and notebooks.
    args, _unknown = parser.parse_known_args()

    rng = np.random.default_rng(args.seed)
    suite_specs = build_suite(args.N, args.eps, rng, args.rel_tol, args.abs_tol)

    results: List[CheckResult] = []
    for spec in suite_specs:
        r = run_check(
            name=spec["name"],
            make_data=spec["make_data"],
            test_fn=spec["test_fn"],
            ref_fn=spec["ref_fn"],
            rel_tol=spec["rel_tol"],
            abs_tol=spec["abs_tol"],
            repeat=args.repeat,
            notes=spec.get("notes", ""),
        )
        results.append(r)

        status = "OK" if r.ok else "FAIL"
        print(f"[{status}] {r.name}  rel_l2={_fmt_float(r.rel_l2)}  max_abs={_fmt_float(r.max_abs)}")

        if args.fail_fast and (not r.ok):
            break

    suite = SuiteResult(results=results)

    print("\nSummary")
    print_table(results)
    print(f"\nPassed: {suite.n_passed}/{suite.n_total}  Failed: {suite.n_failed}/{suite.n_total}")

    if args.json_out is not None:
        payload = {
            "N": args.N,
            "eps": args.eps,
            "rel_tol_default": args.rel_tol,
            "abs_tol_default": args.abs_tol,
            "seed": args.seed,
            "repeat": args.repeat,
            "passed": suite.n_passed,
            "failed": suite.n_failed,
            "results": to_jsonable(results),
        }
        with open(args.json_out, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)
        print(f"Wrote JSON report: {args.json_out}")

    if args.plot:
        plot_results(results, out_prefix=args.plot_prefix)

    if suite.n_failed > 0:
        raise SystemExit(1)


if __name__ == "__main__":
    main()

Repo root: /Users/junaida/Documents/Non-Uniform-QFT
Added to sys.path: /Users/junaida/Documents/Non-Uniform-QFT/src
[OK] uniform: lowrank vs dense  rel_l2=0  max_abs=0
[OK] uniform: dense vs fft  rel_l2=0  max_abs=0
[OK] uniform: lowrank vs fft  rel_l2=0  max_abs=0
[OK] random nodes: lowrank vs dense  rel_l2=4.351e-14  max_abs=1.235e-13
[OK] perturbed uniform: lowrank vs dense  rel_l2=8.940e-14  max_abs=2.949e-13
[OK] impulse: dense vs expected  rel_l2=3.576e-14  max_abs=1.137e-13
[OK] impulse: lowrank vs expected  rel_l2=3.228e-14  max_abs=1.161e-13
[OK] near-colliding: lowrank vs dense  rel_l2=4.427e-14  max_abs=1.227e-13
[OK] clustered: lowrank vs dense  rel_l2=4.383e-14  max_abs=1.431e-13

Summary
 #  |  Test                                 |  PASS  |  rel_l2     |  max_abs    |  rel_tol    |  abs_tol    |  t_test(s)  |  t_ref(s)  |  Notes           
----+---------------------------------------+--------+-------------+-------------+-------------+-------------+-------------+---------